# Test: `load_fed_curves`

Verifies that `treasury_bonds.py` downloads, parses, and saves the Fed GSW dataset correctly.

In [5]:
import sys
sys.path.insert(0, "../src")

from termstructure.ingest.treasury_bonds import load_fed_curves, OUTPUT_PATH

## Check 1 — Download and inspect

In [6]:
df = load_fed_curves()

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nDate range:", df.index.min(), "→", df.index.max())
df.head()

  16.6 MB
Downloaded 16.6 MB. Parsing...
Saved 16,170 rows to C:\Users\aarna\Documents\termstructure\data\processed\treasury_bonds.parquet
Shape: (16170, 36)

Columns: ['beta0', 'beta1', 'beta2', 'beta3', 'tau1', 'tau2', 'sveny01', 'sveny02', 'sveny03', 'sveny04', 'sveny05', 'sveny06', 'sveny07', 'sveny08', 'sveny09', 'sveny10', 'sveny11', 'sveny12', 'sveny13', 'sveny14', 'sveny15', 'sveny16', 'sveny17', 'sveny18', 'sveny19', 'sveny20', 'sveny21', 'sveny22', 'sveny23', 'sveny24', 'sveny25', 'sveny26', 'sveny27', 'sveny28', 'sveny29', 'sveny30']

Dtypes:
 beta0      float64
beta1      float64
beta2      float64
beta3      float64
tau1       float64
tau2       float64
sveny01    float64
sveny02    float64
sveny03    float64
sveny04    float64
sveny05    float64
sveny06    float64
sveny07    float64
sveny08    float64
sveny09    float64
sveny10    float64
sveny11    float64
sveny12    float64
sveny13    float64
sveny14    float64
sveny15    float64
sveny16    float64
sveny17    float64
sv

,beta0,beta1,beta2,beta3,tau1,tau2,sveny01,sveny02,sveny03,sveny04,...,sveny21,sveny22,sveny23,sveny24,sveny25,sveny26,sveny27,sveny28,sveny29,sveny30
date,,,,,,,,,,,,,,,,,,,,,
1961-06-14,3.917606,-1.277955,-1.949397,0.0,0.339218,NaN,2.9825,3.3771,3.5530,3.6439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-06-15,3.978498,-1.257404,-2.247617,0.0,0.325775,NaN,2.9941,3.4137,3.5981,3.6930,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-06-16,3.984350,-1.429538,-1.885024,0.0,0.348817,NaN,3.0012,3.4142,3.5994,3.6953,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-06-19,4.004379,-0.723311,-3.310743,0.0,0.282087,NaN,2.9949,3.4386,3.6252,3.7199,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-06-20,3.985789,-0.900432,-2.844809,0.0,0.310316,NaN,2.9833,3.4101,3.5986,3.6952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Check 2 — Svensson parameters look reasonable

beta0 is the long-run asymptotic rate. The data goes back to 1961 and covers the
early-1980s rate spike when US yields hit ~16%, so beta0 can legitimately exceed 20%.
We just check it's finite and positive. tau1 must always be positive.

In [7]:
params = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]
print("Svensson parameter stats:")
print(df[params].describe().round(3))

assert (df["beta0"].dropna() > 0).all(), "beta0 should be positive"
assert (df["tau1"].dropna() > 0).all(), "tau1 should be positive"
print(f"\nbeta0 range: {df['beta0'].min():.2f}% → {df['beta0'].max():.2f}%")
print("Parameter sanity checks passed.")

Svensson parameter stats:
           beta0      beta1       beta2       beta3       tau1       tau2
count  16170.000  16170.000   16170.000   16170.000  16170.000  11550.000
mean       5.507     -0.737    -292.329     294.709      2.427      9.319
std        4.489      4.862    5325.114    5324.902      3.308      7.511
min        0.000    -39.730 -340683.769  -10524.663      0.100      0.100
25%        2.618     -2.925      -5.756       0.000      0.685      3.679
50%        4.559     -1.011      -0.963       2.692      1.532      9.243
75%        7.703      1.622       1.638      16.778      2.664     13.397
max       25.000     97.176   10523.578  340681.503     30.000    180.859

beta0 range: 0.00% → 25.00%
Parameter sanity checks passed.


## Check 3 — Zero yield panel is complete

sveny01–sveny30 should have very few NaNs (only on holidays).
Yields should be in a plausible range (0–20%).

In [8]:
zero_cols = [f"sveny{i:02d}" for i in range(1, 31)]
zero_panel = df[zero_cols]

nan_per_col = zero_panel.isna().mean().round(3)
print("NaN rate per column:")
print(nan_per_col.to_string())

ymin = zero_panel.min().min()
ymax = zero_panel.max().max()
print(f"\nYield range: {ymin:.2f}% → {ymax:.2f}%")

NaN rate per column:
sveny01    0.000
sveny02    0.000
sveny03    0.000
sveny04    0.000
sveny05    0.000
sveny06    0.000
sveny07    0.000
sveny08    0.157
sveny09    0.157
sveny10    0.157
sveny11    0.161
sveny12    0.161
sveny13    0.161
sveny14    0.161
sveny15    0.161
sveny16    0.309
sveny17    0.309
sveny18    0.309
sveny19    0.309
sveny20    0.309
sveny21    0.377
sveny22    0.377
sveny23    0.377
sveny24    0.377
sveny25    0.377
sveny26    0.377
sveny27    0.377
sveny28    0.377
sveny29    0.377
sveny30    0.377

Yield range: 0.06% → 16.46%


## Check 4 — Spot-check against known Fed values

On 2023-01-03, the Fed's published beta0 should be around 4.0–4.5
and the 10-year zero yield (sveny10) should be close to the CMT DGS10 value of 3.79%.

In [9]:
row = df.loc["2023-01-03", ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2", "sveny10"]]
print(row)

sveny10 = row["sveny10"]
assert abs(sveny10 - 3.79) < 0.2, f"sveny10 on 2023-01-03 expected ~3.79%, got {sveny10:.2f}%"
print(f"\nSpot check passed: 10Y zero yield on 2023-01-03 = {sveny10:.2f}%")

beta0        0.000043
beta1        5.031375
beta2     -125.872907
beta3      131.460222
tau1         9.187250
tau2         9.787652
sveny10      3.776000
Name: 2023-01-03 00:00:00, dtype: float64

Spot check passed: 10Y zero yield on 2023-01-03 = 3.78%


## Check 5 — Parquet round-trip

In [10]:
import pandas as pd

assert OUTPUT_PATH.exists(), f"Parquet not found at {OUTPUT_PATH}"
df_disk = pd.read_parquet(OUTPUT_PATH)

print("Saving to:", OUTPUT_PATH)
print("In-memory shape:", df.shape)
print("On-disk shape:  ", df_disk.shape)

assert df_disk.shape == (len(df), len(df.columns) + 1), "Unexpected shape on disk"
print("\nRound-trip check passed.")

Saving to: C:\Users\aarna\Documents\termstructure\data\processed\treasury_bonds.parquet
In-memory shape: (16170, 36)
On-disk shape:   (16170, 37)

Round-trip check passed.
